In [2]:
# ===== HEADER: loader + helpers =====
import time, itertools, random
import pandas as pd
import numpy as np

def load_ctx_drug(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def pairwise_tissue_sets(df: pd.DataFrame):
    """Return drugs, tissues, and S[a][b] = set of tissues where a>b."""
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    for i, a in enumerate(drugs):
        for j, b in enumerate(drugs):
            if i==j: continue
            wins = set()
            for t in tissues:
                va, vb = df.at[t, a], df.at[t, b]
                if pd.isna(va) or pd.isna(vb):
                    continue
                if va > vb:
                    wins.add(t)
            S[i][j] = wins
    return drugs, tissues, S

def path_intersection_prefix(order_idxs, S, lam):
    """Given index order and S[i][j], return the longest λ-consistent prefix path and its common tissues S*."""
    if not order_idxs: return [], set(), []
    S_common = None
    adj_sets = []
    path = [order_idxs[0]]
    for i,j in zip(order_idxs[:-1], order_idxs[1:]):
        Sab = S[i][j]
        S_common = Sab if S_common is None else (S_common & Sab)
        if len(S_common) >= lam:
            path.append(j)
            adj_sets.append(Sab & S_common)
        else:
            break
    return path, (S_common if S_common is not None else set()), adj_sets

def print_result(tag, elapsed_s, order, S):
    print(f"[{tag}] time = {elapsed_s:.3f} s | path length = {len(order)} | |S| = {len(S)}")
    if order:
        print("  Path:", " > ".join(order))
    else:
        print("  Path: (empty)")


In [3]:
# ============ TA-Schulze (λ-path) with SOFT DEADLINE + Benchmarks ============

def ta_schulze_path_soft(df: pd.DataFrame, lam: int, time_budget_s: float = 3600.0):
    """
    Soft-deadline version of ta_schulze_path:
    - Checks time during matrix updates and DFS.
    - Returns best-so-far if time budget exceeded.
    """
    t0 = time.time()
    drugs, tissues, S = pairwise_tissue_sets(df)
    n = len(drugs)

    # Step 1: Compute P[i][j] = strength of strongest tissue-consistent path
    P = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                P[i][j] = len(S[i][j])

    for k in range(n):
        if time.time() - t0 > time_budget_s:
            return [], set(), time.time() - t0, "deadline"
        for i in range(n):
            if i == k:
                continue
            for j in range(n):
                if j in (i, k):
                    continue
                P[i][j] = max(P[i][j], min(P[i][k], P[k][j]))

    # Step 2: Build λ-beat graph
    G = [[False]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j and P[i][j] >= lam and P[i][j] > P[j][i]:
                G[i][j] = True

    # Step 3: DFS for longest λ-consistent path
    best_path_idx, best_S = [], set()
    deadline_hit = False
    for start in range(n):
        if time.time() - t0 > time_budget_s:
            deadline_hit = True
            break
        stack = [(start, [start], None)]
        while stack:
            if time.time() - t0 > time_budget_s:
                deadline_hit = True
                break
            cur, path, S_common = stack.pop()
            for nxt in range(n):
                if not G[cur][nxt] or nxt in path:
                    continue
                Sab = S[cur][nxt]
                new_S = Sab if S_common is None else (S_common & Sab)
                if len(new_S) >= lam:
                    new_path = path + [nxt]
                    stack.append((nxt, new_path, new_S))
                    if len(new_path) > len(best_path_idx):
                        best_path_idx, best_S = new_path, new_S
        if deadline_hit:
            break

    elapsed = time.time() - t0
    order = [drugs[i] for i in best_path_idx]
    status = "deadline" if deadline_hit else "ok"
    return order, best_S, elapsed, status


# ---------------- Benchmark 1: λ-increasing on ALL drugs ----------------
def tasch_lambda_increasing(csv_path="efficacy.csv",
                            lam_start=5, lam_end=60, lam_step=5,
                            time_budget_s=3600,
                            out_xlsx="tasch_lambda_increasing.xlsx"):
    """
    Runs TA-Schulze-PATH on all drugs, increasing λ each step (soft deadline per λ).
    """
    df_full = load_ctx_drug(csv_path)
    n_drugs = df_full.shape[1]
    rows = []
    print(f"TA-Schulze λ-sweep on ALL {n_drugs} drugs (soft deadline per λ = {time_budget_s//60:.0f} min)")

    for lam in range(lam_start, lam_end + 1, lam_step):
        order, S_common, elapsed, status = ta_schulze_path_soft(df_full, lam, time_budget_s=time_budget_s)
        rows.append({
            "method": "TA-Schulze-PATH",
            "n_drugs": n_drugs,
            "lambda": lam,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  λ={lam:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping λ sweep.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ------------- Benchmark 2: drugs-increasing (fixed λ) ----------------
def tasch_drug_increasing(csv_path="efficacy.csv",
                          lam_fixed=30,
                          start_size=5, step=5, seed=42,
                          time_budget_s=3600,
                          out_xlsx="tasch_drug_increasing.xlsx"):
    """
    Runs TA-Schulze-PATH while gradually increasing #drugs (fixed λ).
    Nested subsets (same seed). Soft deadline per step.
    """
    df_full = load_ctx_drug(csv_path)
    n_total = df_full.shape[1]
    sizes = list(range(start_size, n_total + 1, step))
    rows = []

    rng = random.Random(seed)
    cols = list(df_full.columns)
    rng.shuffle(cols)

    print(f"TA-Schulze drug-sweep at λ={lam_fixed}; total drugs = {n_total} (soft deadline per size = {time_budget_s//60:.0f} min)")
    for n in sizes:
        sub = df_full[cols[:n]]
        order, S_common, elapsed, status = ta_schulze_path_soft(sub, lam_fixed, time_budget_s=time_budget_s)
        rows.append({
            "method": "TA-Schulze-PATH",
            "n_drugs": n,
            "lambda": lam_fixed,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  n={n:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping size growth.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ---------------------- Example calls (uncomment to run) ----------------------
# tasch_lambda_increasing("efficacy.csv",
#                         lam_start=3, lam_end=60, lam_step=3,
#                         time_budget_s=3600,
#                         out_xlsx="tasch_lambda_increasing.xlsx")

# tasch_drug_increasing("efficacy.csv",
#                       lam_fixed=30,
#                       start_size=5, step=5, seed=42,
#                       time_budget_s=3600,
#                       out_xlsx="tasch_drug_increasing.xlsx")


In [ ]:
tasch_drug_increasing("efficacy.csv",
                       lam_fixed=30,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m.xlsx")


TA-Schulze drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.04 s | len=7
  n= 15 | status=ok       | time=0.10 s | len=9
  n= 20 | status=ok       | time=0.25 s | len=10
  n= 25 | status=ok       | time=0.69 s | len=11
  n= 30 | status=ok       | time=1.41 s | len=11
  n= 35 | status=ok       | time=4.95 s | len=12
  n= 40 | status=ok       | time=28.69 s | len=13
  n= 45 | status=ok       | time=89.57 s | len=13
  n= 50 | status=ok       | time=221.52 s | len=13
  n= 55 | status=ok       | time=483.01 s | len=13
  n= 60 | status=ok       | time=970.12 s | len=14
  n= 65 | status=deadline | time=1200.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,30,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.008774,ok
1,TA-Schulze-PATH,10,30,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,7,0.037590,ok
2,TA-Schulze-PATH,15,30,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > T...,9,0.102695,ok
3,TA-Schulze-PATH,20,30,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > H...,10,0.251899,ok
4,TA-Schulze-PATH,25,30,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,11,0.690520,ok
5,TA-Schulze-PATH,30,30,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > P...,11,1.405624,ok
6,TA-Schulze-PATH,35,30,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,12,4.949035,ok
7,TA-Schulze-PATH,40,30,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > MECH...,13,28.688225,ok
8,TA-Schulze-PATH,45,30,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,13,89.572083,ok
9,TA-Schulze-PATH,50,30,THEOPHYLLINE > SANGUINARINE SULFATE > CURCUMIN...,13,221.515558,ok


In [ ]:
 tasch_lambda_increasing("efficacy.csv",
                         lam_start=5, lam_end=49, lam_step=5,
                         time_budget_s=120,
                         out_xlsx="tasch_lambda_increasing.xlsx")

TA-Schulze λ-sweep on ALL 273 drugs (soft deadline per λ = 2 min)
  λ=  5 | status=deadline | time=120.00 s | len=17
  ⏱️ Soft deadline hit; stopping λ sweep.
✅ Saved -> tasch_lambda_increasing.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,273,5,ASPIRIN > BUMETANIDE > NITAZOXANIDE > METERGOL...,17,120.000005,deadline


In [ ]:
tasch_drug_increasing("efficacy.csv",
                       lam_fixed=35,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasingL3530m.xlsx")

TA-Schulze drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.02 s | len=6
  n= 15 | status=ok       | time=0.06 s | len=8
  n= 20 | status=ok       | time=0.12 s | len=9
  n= 25 | status=ok       | time=0.26 s | len=10
  n= 30 | status=ok       | time=0.47 s | len=10
  n= 35 | status=ok       | time=1.31 s | len=11
  n= 40 | status=ok       | time=5.20 s | len=12
  n= 45 | status=ok       | time=15.04 s | len=12
  n= 50 | status=ok       | time=33.73 s | len=12
  n= 55 | status=ok       | time=66.88 s | len=12
  n= 60 | status=ok       | time=124.75 s | len=13
  n= 65 | status=ok       | time=236.09 s | len=13
  n= 70 | status=ok       | time=307.40 s | len=13
  n= 75 | status=ok       | time=485.98 s | len=13
  n= 80 | status=ok       | time=799.19 s | len=13
  n= 85 | status=deadline | time=1200.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_incr

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,35,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.005570,ok
1,TA-Schulze-PATH,10,35,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,6,0.023490,ok
2,TA-Schulze-PATH,15,35,PD-0166285 > MESALAMINE > TRAZODONE > ZIPRASID...,8,0.059337,ok
3,TA-Schulze-PATH,20,35,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MES...,9,0.123902,ok
4,TA-Schulze-PATH,25,35,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > H...,10,0.258242,ok
5,TA-Schulze-PATH,30,35,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MEC...,10,0.470052,ok
6,TA-Schulze-PATH,35,35,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,11,1.306905,ok
7,TA-Schulze-PATH,40,35,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > ARIP...,12,5.204149,ok
8,TA-Schulze-PATH,45,35,THEOPHYLLINE > DEXAMETHASONE > STREPTOZOCIN > ...,12,15.043768,ok
9,TA-Schulze-PATH,50,35,THEOPHYLLINE > HALOPERIDOL DECANOATE > MECHLOR...,12,33.734762,ok


In [ ]:
tasch_drug_increasing("efficacy.csv",
                       lam_fixed=25,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasingL2530m.xlsx")

TA-Schulze drug-sweep at λ=25; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.03 s | len=7
  n= 15 | status=ok       | time=0.07 s | len=10
  n= 20 | status=ok       | time=0.17 s | len=11
  n= 25 | status=ok       | time=0.60 s | len=11
  n= 30 | status=ok       | time=1.43 s | len=11
  n= 35 | status=ok       | time=5.94 s | len=12
  n= 40 | status=ok       | time=33.89 s | len=13
  n= 45 | status=ok       | time=122.02 s | len=14
  n= 50 | status=ok       | time=331.27 s | len=14
  n= 55 | status=ok       | time=794.80 s | len=14
  n= 60 | status=deadline | time=1200.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasingL2530m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,25,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.005575,ok
1,TA-Schulze-PATH,10,25,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,7,0.025521,ok
2,TA-Schulze-PATH,15,25,THEOPHYLLINE > DEXAMETHASONE > PD-0166285 > IL...,10,0.068244,ok
3,TA-Schulze-PATH,20,25,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,11,0.174556,ok
4,TA-Schulze-PATH,25,25,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > L...,11,0.597289,ok
5,TA-Schulze-PATH,30,25,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MEC...,11,1.426357,ok
6,TA-Schulze-PATH,35,25,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,12,5.941520,ok
7,TA-Schulze-PATH,40,25,THEOPHYLLINE > ITRACONAZOLE > CURCUMIN > PF-04...,13,33.894297,ok
8,TA-Schulze-PATH,45,25,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > MECH...,14,122.019227,ok
9,TA-Schulze-PATH,50,25,THEOPHYLLINE > SANGUINARINE SULFATE > CURCUMIN...,14,331.268812,ok


In [ ]:
tasch_drug_increasing("efficacy-diabities.csv",
                       lam_fixed=30,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diabities.xlsx")


TA-Schulze drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=5
  n= 10 | status=ok       | time=0.03 s | len=7
  n= 15 | status=ok       | time=0.09 s | len=7
  n= 20 | status=ok       | time=0.21 s | len=9
  n= 25 | status=ok       | time=0.66 s | len=10
  n= 30 | status=ok       | time=2.58 s | len=10
  n= 35 | status=ok       | time=3.77 s | len=11
  n= 40 | status=ok       | time=21.62 s | len=12
  n= 45 | status=ok       | time=98.96 s | len=13
  n= 50 | status=ok       | time=219.81 s | len=13
  n= 55 | status=ok       | time=412.56 s | len=13
  n= 60 | status=ok       | time=849.40 s | len=13
  n= 65 | status=deadline | time=1200.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_diabities.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,30,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,0.009213,ok
1,TA-Schulze-PATH,10,30,DACTINOMYCIN > WORTMANNIN > RITUXIMAB > LEUCOV...,7,0.034112,ok
2,TA-Schulze-PATH,15,30,DACTINOMYCIN > MIDAZOLAM HYDROCHLORIDE > RITUX...,7,0.085576,ok
3,TA-Schulze-PATH,20,30,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > D...,9,0.212198,ok
4,TA-Schulze-PATH,25,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > PREGNANT...,10,0.659138,ok
5,TA-Schulze-PATH,30,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,10,2.583832,ok
6,TA-Schulze-PATH,35,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,11,3.769168,ok
7,TA-Schulze-PATH,40,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,12,21.615247,ok
8,TA-Schulze-PATH,45,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,13,98.960485,ok
9,TA-Schulze-PATH,50,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,13,219.808037,ok


In [ ]:
tasch_drug_increasing("efficacy-diabities.csv",
                       lam_fixed=35,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diabities_l35.xlsx")


TA-Schulze drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.02 s | len=4
  n= 10 | status=ok       | time=0.07 s | len=7
  n= 15 | status=ok       | time=0.16 s | len=7
  n= 20 | status=ok       | time=0.34 s | len=8
  n= 25 | status=ok       | time=0.48 s | len=9
  n= 30 | status=ok       | time=0.77 s | len=9
  n= 35 | status=ok       | time=1.66 s | len=10
  n= 40 | status=ok       | time=6.58 s | len=11
  n= 45 | status=ok       | time=28.38 s | len=12
  n= 50 | status=ok       | time=60.39 s | len=12
  n= 55 | status=ok       | time=105.24 s | len=12
  n= 60 | status=ok       | time=209.54 s | len=12
  n= 65 | status=ok       | time=344.48 s | len=12
  n= 70 | status=ok       | time=485.77 s | len=12
  n= 75 | status=ok       | time=971.58 s | len=12
  n= 80 | status=deadline | time=1200.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_diabities_l35.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,35,CARBOPLATIN > CHEMBL:CHEMBL493863 > FULVESTRAN...,4,0.017222,ok
1,TA-Schulze-PATH,10,35,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > CA...,7,0.070388,ok
2,TA-Schulze-PATH,15,35,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > CA...,7,0.158916,ok
3,TA-Schulze-PATH,20,35,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > D...,8,0.343089,ok
4,TA-Schulze-PATH,25,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,9,0.482479,ok
5,TA-Schulze-PATH,30,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,9,0.774902,ok
6,TA-Schulze-PATH,35,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,10,1.659374,ok
7,TA-Schulze-PATH,40,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,11,6.582482,ok
8,TA-Schulze-PATH,45,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,12,28.382436,ok
9,TA-Schulze-PATH,50,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,12,60.386434,ok


In [ ]:
tasch_drug_increasing("efficacy-diabities.csv",
                       lam_fixed=25,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diabities_l25.xlsx")


TA-Schulze drug-sweep at λ=25; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=5
  n= 10 | status=ok       | time=0.03 s | len=8
  n= 15 | status=ok       | time=0.10 s | len=8
  n= 20 | status=ok       | time=0.23 s | len=9
  n= 25 | status=ok       | time=0.89 s | len=11
  n= 30 | status=ok       | time=4.04 s | len=12
  n= 35 | status=ok       | time=8.77 s | len=12
  n= 40 | status=ok       | time=57.71 s | len=13
  n= 45 | status=ok       | time=295.19 s | len=14
  n= 50 | status=ok       | time=721.43 s | len=14
  n= 55 | status=deadline | time=1200.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_diabities_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,25,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,0.008376,ok
1,TA-Schulze-PATH,10,25,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > WO...,8,0.034611,ok
2,TA-Schulze-PATH,15,25,CYCLOPHOSPHAMIDE ANHYDROUS > MIDAZOLAM HYDROCH...,8,0.103754,ok
3,TA-Schulze-PATH,20,25,SULFASALAZINE > OXIDOPAMINE HYDROCHLORIDE > CA...,9,0.229855,ok
4,TA-Schulze-PATH,25,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > PREGNANT...,11,0.888663,ok
5,TA-Schulze-PATH,30,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,12,4.044047,ok
6,TA-Schulze-PATH,35,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,12,8.766336,ok
7,TA-Schulze-PATH,40,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,13,57.706192,ok
8,TA-Schulze-PATH,45,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,14,295.192523,ok
9,TA-Schulze-PATH,50,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,14,721.433422,ok


In [3]:
tasch_drug_increasing("efficacy-bpco.csv",
                       lam_fixed=25,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_bpco_l25.xlsx")

TA-Schulze drug-sweep at λ=25; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.02 s | len=3
  n= 10 | status=ok       | time=0.07 s | len=6
  n= 15 | status=ok       | time=0.16 s | len=6
  n= 20 | status=ok       | time=0.34 s | len=8
  n= 25 | status=ok       | time=0.78 s | len=9
  n= 30 | status=ok       | time=1.88 s | len=10
  n= 35 | status=ok       | time=4.53 s | len=12
  n= 40 | status=ok       | time=12.51 s | len=12
  n= 45 | status=ok       | time=52.76 s | len=13
  n= 50 | status=ok       | time=196.77 s | len=14
  n= 55 | status=ok       | time=347.96 s | len=14
  n= 60 | status=deadline | time=1200.00 s | len=15
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_bpco_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,25,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.017408,ok
1,TA-Schulze-PATH,10,25,THEOPHYLLINE > SORAFENIB > FULVESTRANT > ALPHA...,6,0.065490,ok
2,TA-Schulze-PATH,15,25,THEOPHYLLINE > OLANZAPINE > INTERLEUKIN-11 > A...,6,0.160885,ok
3,TA-Schulze-PATH,20,25,THEOPHYLLINE > OLANZAPINE > DICLOFENAC SODIUM ...,8,0.339170,ok
4,TA-Schulze-PATH,25,25,THEOPHYLLINE > DICLOFENAC SODIUM > SORAFENIB >...,9,0.778795,ok
5,TA-Schulze-PATH,30,25,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,10,1.879817,ok
6,TA-Schulze-PATH,35,25,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,12,4.527247,ok
7,TA-Schulze-PATH,40,25,PAXALISIB > INSULIN > DICLOFENAC SODIUM > VORI...,12,12.511912,ok
8,TA-Schulze-PATH,45,25,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,13,52.761125,ok
9,TA-Schulze-PATH,50,25,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,14,196.765168,ok


In [4]:
tasch_drug_increasing("efficacy-bpco.csv",
                       lam_fixed=30,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_bpco_l30.xlsx")

TA-Schulze drug-sweep at λ=30; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.04 s | len=5
  n= 15 | status=ok       | time=0.08 s | len=6
  n= 20 | status=ok       | time=0.16 s | len=7
  n= 25 | status=ok       | time=0.28 s | len=8
  n= 30 | status=ok       | time=0.55 s | len=10
  n= 35 | status=ok       | time=1.18 s | len=11
  n= 40 | status=ok       | time=2.12 s | len=11
  n= 45 | status=ok       | time=8.58 s | len=12
  n= 50 | status=ok       | time=27.79 s | len=13
  n= 55 | status=ok       | time=38.94 s | len=13
  n= 60 | status=ok       | time=145.23 s | len=14
  n= 65 | status=ok       | time=270.26 s | len=15
  n= 70 | status=ok       | time=665.96 s | len=16
  n= 75 | status=deadline | time=1200.00 s | len=16
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_bpco_l30.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,30,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.008106,ok
1,TA-Schulze-PATH,10,30,THEOPHYLLINE > SORAFENIB > ALPHA-TOCOPHEROL > ...,5,0.035180,ok
2,TA-Schulze-PATH,15,30,OLANZAPINE > SORAFENIB > INTERLEUKIN-11 > ALPH...,6,0.080050,ok
3,TA-Schulze-PATH,20,30,THEOPHYLLINE > DICLOFENAC SODIUM > INTERLEUKIN...,7,0.158687,ok
4,TA-Schulze-PATH,25,30,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,8,0.283253,ok
5,TA-Schulze-PATH,30,30,PAXALISIB > CAPECITABINE > SORAFENIB > CARBAMA...,10,0.553279,ok
6,TA-Schulze-PATH,35,30,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,11,1.178771,ok
7,TA-Schulze-PATH,40,30,PAXALISIB > INSULIN > DICLOFENAC SODIUM > VORI...,11,2.123752,ok
8,TA-Schulze-PATH,45,30,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,12,8.583490,ok
9,TA-Schulze-PATH,50,30,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,13,27.792633,ok


In [5]:
tasch_drug_increasing("efficacy-bpco.csv",
                       lam_fixed=35,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_bpco_l35.xlsx")

TA-Schulze drug-sweep at λ=35; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.03 s | len=5
  n= 15 | status=ok       | time=0.08 s | len=5
  n= 20 | status=ok       | time=0.17 s | len=7
  n= 25 | status=ok       | time=0.25 s | len=8
  n= 30 | status=ok       | time=0.44 s | len=9
  n= 35 | status=ok       | time=1.09 s | len=10
  n= 40 | status=ok       | time=2.03 s | len=10
  n= 45 | status=ok       | time=3.13 s | len=11
  n= 50 | status=ok       | time=10.00 s | len=12
  n= 55 | status=ok       | time=12.98 s | len=12
  n= 60 | status=ok       | time=42.09 s | len=13
  n= 65 | status=ok       | time=74.60 s | len=14
  n= 70 | status=ok       | time=171.55 s | len=15
  n= 75 | status=ok       | time=337.35 s | len=15
  n= 80 | status=ok       | time=599.81 s | len=15
  n= 85 | status=ok       | time=984.09 s | len=15
  n= 90 | status=deadline | time=1200.00 s | len=15
  ⏱️ Soft deadline hit; st

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,35,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.007931,ok
1,TA-Schulze-PATH,10,35,THEOPHYLLINE > SORAFENIB > ALPHA-TOCOPHEROL > ...,5,0.033246,ok
2,TA-Schulze-PATH,15,35,OSIMERTINIB > INTERLEUKIN-11 > ALPHA-TOCOPHERO...,5,0.080874,ok
3,TA-Schulze-PATH,20,35,THEOPHYLLINE > DICLOFENAC SODIUM > INTERLEUKIN...,7,0.168649,ok
4,TA-Schulze-PATH,25,35,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,8,0.249361,ok
5,TA-Schulze-PATH,30,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,9,0.436247,ok
6,TA-Schulze-PATH,35,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,10,1.086787,ok
7,TA-Schulze-PATH,40,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,10,2.030651,ok
8,TA-Schulze-PATH,45,35,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,11,3.127705,ok
9,TA-Schulze-PATH,50,35,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,12,9.995993,ok


In [5]:
tasch_drug_increasing("efficacy-diabetes.csv",
                       lam_fixed=35,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diab_l35.xlsx")

TA-Schulze drug-sweep at λ=35; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.06 s | len=6
  n= 15 | status=ok       | time=0.16 s | len=8
  n= 20 | status=ok       | time=0.31 s | len=8
  n= 25 | status=ok       | time=0.59 s | len=8
  n= 30 | status=ok       | time=0.87 s | len=9
  n= 35 | status=ok       | time=1.45 s | len=9
  n= 40 | status=ok       | time=2.77 s | len=9
  n= 45 | status=ok       | time=9.04 s | len=10
  n= 50 | status=ok       | time=16.50 s | len=10
  n= 55 | status=ok       | time=23.37 s | len=10
  n= 60 | status=ok       | time=47.19 s | len=10
  n= 65 | status=ok       | time=85.31 s | len=11
  n= 70 | status=ok       | time=140.74 s | len=11
  n= 75 | status=ok       | time=217.78 s | len=12
  n= 80 | status=ok       | time=281.11 s | len=12
  n= 85 | status=ok       | time=407.12 s | len=12
  n= 90 | status=ok       | time=1074.51 s | len=13
  n= 95 | status=deadline | 

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,35,HUMAN > GSK-269962A > BROMOCRIPTINE,3,0.013462,ok
1,TA-Schulze-PATH,10,35,ATENOLOL > HUMAN > GSK-269962A > ANTISENSE OLI...,6,0.061330,ok
2,TA-Schulze-PATH,15,35,ASPIRIN > ATENOLOL > GSK-269962A > ANTISENSE O...,8,0.159629,ok
3,TA-Schulze-PATH,20,35,ASPIRIN > ATENOLOL > GSK-269962A > ANTISENSE O...,8,0.313394,ok
4,TA-Schulze-PATH,25,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,8,0.594016,ok
5,TA-Schulze-PATH,30,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,9,0.874737,ok
6,TA-Schulze-PATH,35,35,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > RG-1530 ...,9,1.448790,ok
7,TA-Schulze-PATH,40,35,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > BORTEZOM...,9,2.771746,ok
8,TA-Schulze-PATH,45,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > METHOTREXATE...,10,9.039744,ok
9,TA-Schulze-PATH,50,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,10,16.502866,ok


In [6]:
tasch_drug_increasing("efficacy-diabetes.csv",
                       lam_fixed=30,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diab_l30.xlsx")

TA-Schulze drug-sweep at λ=30; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.03 s | len=6
  n= 15 | status=ok       | time=0.09 s | len=8
  n= 20 | status=ok       | time=0.21 s | len=8
  n= 25 | status=ok       | time=0.50 s | len=9
  n= 30 | status=ok       | time=1.38 s | len=10
  n= 35 | status=ok       | time=4.92 s | len=10
  n= 40 | status=ok       | time=8.64 s | len=10
  n= 45 | status=ok       | time=23.84 s | len=11
  n= 50 | status=ok       | time=56.80 s | len=11
  n= 55 | status=ok       | time=97.37 s | len=11
  n= 60 | status=ok       | time=213.50 s | len=11
  n= 65 | status=ok       | time=404.56 s | len=12
  n= 70 | status=ok       | time=714.64 s | len=12
  n= 75 | status=ok       | time=1185.68 s | len=12
  n= 80 | status=deadline | time=1200.00 s | len=12
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_diab_l30.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,30,HUMAN > GSK-269962A > BROMOCRIPTINE,3,0.007838,ok
1,TA-Schulze-PATH,10,30,ATENOLOL > HUMAN > RESVERATROL > ANTISENSE OLI...,6,0.032931,ok
2,TA-Schulze-PATH,15,30,ASPIRIN > ATENOLOL > ADALIMUMAB-ADBM > ANTISEN...,8,0.089623,ok
3,TA-Schulze-PATH,20,30,ASPIRIN > ATENOLOL > CYCLOPHOSPHAMIDE ANHYDROU...,8,0.213625,ok
4,TA-Schulze-PATH,25,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,9,0.502340,ok
5,TA-Schulze-PATH,30,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,10,1.375731,ok
6,TA-Schulze-PATH,35,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOXORUBICIN ...,10,4.917231,ok
7,TA-Schulze-PATH,40,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > BORTEZOMIB >...,10,8.639283,ok
8,TA-Schulze-PATH,45,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > METHOTREXATE...,11,23.839376,ok
9,TA-Schulze-PATH,50,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,11,56.798492,ok


In [7]:
tasch_drug_increasing("efficacy-diabetes.csv",
                       lam_fixed=25,
                       start_size=5, step=5, seed=42,
                       time_budget_s=1200,
                       out_xlsx="tasch_drug_increasing30m_diab_l25.xlsx")

TA-Schulze drug-sweep at λ=25; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.03 s | len=6
  n= 15 | status=ok       | time=0.09 s | len=9
  n= 20 | status=ok       | time=0.29 s | len=9
  n= 25 | status=ok       | time=0.75 s | len=10
  n= 30 | status=ok       | time=3.74 s | len=11
  n= 35 | status=ok       | time=8.57 s | len=11
  n= 40 | status=ok       | time=23.95 s | len=11
  n= 45 | status=ok       | time=65.67 s | len=12
  n= 50 | status=ok       | time=173.71 s | len=12
  n= 55 | status=ok       | time=324.75 s | len=12
  n= 60 | status=ok       | time=757.33 s | len=13
  n= 65 | status=deadline | time=1200.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tasch_drug_increasing30m_diab_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Schulze-PATH,5,25,ATENOLOL > HUMAN > GSK-269962A > BROMOCRIPTINE,4,0.007921,ok
1,TA-Schulze-PATH,10,25,HUMAN > GSK-269962A > ANTISENSE OLIGONUCLEOTID...,6,0.034192,ok
2,TA-Schulze-PATH,15,25,ASPIRIN > ATENOLOL > GSK-269962A > BROMOCRIPTI...,9,0.094151,ok
3,TA-Schulze-PATH,20,25,ASPIRIN > ATENOLOL > CYCLOPHOSPHAMIDE ANHYDROU...,9,0.286273,ok
4,TA-Schulze-PATH,25,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,10,0.751959,ok
5,TA-Schulze-PATH,30,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,11,3.735308,ok
6,TA-Schulze-PATH,35,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOXORUBICIN ...,11,8.571859,ok
7,TA-Schulze-PATH,40,25,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > BORTEZOM...,11,23.954033,ok
8,TA-Schulze-PATH,45,25,ASPIRIN > ATENOLOL > METHOTREXATE > BORTEZOMIB...,12,65.668381,ok
9,TA-Schulze-PATH,50,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,12,173.713161,ok
